# 02 · No supervisado: ¿existen tipos de países?
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrezconz/investigacion-aumentada-ml/blob/main/notebooks/02_no_supervisado_kmeans_paises.ipynb)

**Pregunta:** si describimos 193 países con 21 variables, ¿se forman grupos sin que nadie se los diga al modelo?

**Familia:** no supervisado. No hay respuesta conocida: el modelo busca estructura.

**Cómo funciona K-Means:**
1. Escoge *k* centros al azar.
2. Asigna cada país al centro más cercano.
3. Mueve cada centro al promedio de sus países.
4. Repite hasta que los grupos dejan de cambiar.

**PCA** (componentes principales) resume las 21 variables en dos ejes para poder dibujar los países en un plano.

In [ ]:
# Preparación: trae los datos del repositorio (funciona igual en Google Colab)
import os, urllib.request
REPO = "https://raw.githubusercontent.com/andrezconz/investigacion-aumentada-ml/main/datos/"

def traer(nombre):
    local = os.path.join("..", "datos", nombre)
    if os.path.exists(local):
        return local
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(REPO + nombre, nombre)
    return nombre

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(traer("desarrollo_paises_real.csv"))
variables = ["gdp_per_capita_usd", "life_expectancy", "secondary_enrollment_pct", "tertiary_enrollment_pct",
             "adult_literacy_pct", "gini_index", "internet_users_pct", "co2_per_capita", "unemployment_pct",
             "urban_pop_pct", "health_exp_pct_gdp", "educ_exp_pct_gdp", "manufacturing_pct_gdp", "fertility_rate",
             "under5_mortality", "wgi_control_corruption", "wgi_rule_of_law", "wgi_govt_effectiveness",
             "wgi_regulatory_quality", "wgi_political_stability", "wgi_voice_accountability"]
print(len(df), "países ·", len(variables), "variables")
df[["country", "region"] + variables[:4]].head()

> **Tip:** el Índice de Desarrollo Humano (IDH) se deja **por fuera** a propósito. Lo usamos al final para revisar si los grupos tienen sentido. Si lo metemos al modelo, el resultado se vuelve circular.

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df[variables].copy()
X = X.fillna(X.median())                       # faltantes: mediana (decisión explícita, no oculta)
for v in ["gdp_per_capita_usd", "co2_per_capita"]:
    X[v] = np.log1p(X[v])                      # variables muy desiguales: escala logarítmica
Xz = StandardScaler().fit_transform(X)         # todas en la misma escala
print("Faltantes imputados:", int(df[variables].isna().sum().sum()))

## ¿Cuántos grupos?
El **coeficiente de silueta** mide qué tan separados quedan los grupos (más alto es mejor, entre −1 y 1).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=25, random_state=42).fit(Xz)
    print(f"k = {k}  ·  silueta = {silhouette_score(Xz, km.labels_):.3f}")

In [ ]:
k = 4   # prueba cambiar este número
km = KMeans(n_clusters=k, n_init=25, random_state=42).fit(Xz)
df["grupo"] = km.labels_ + 1
df["grupo"].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(Xz)
P = pca.transform(Xz)
plt.figure(figsize=(11, 7))
for g in sorted(df["grupo"].unique()):
    m = df["grupo"] == g
    plt.scatter(P[m, 0], P[m, 1], s=40, alpha=.75, label=f"Grupo {g}")
for pais in ["Colombia", "Norway", "Nigeria", "China", "Costa Rica", "Ecuador"]:
    i = df.index[df["country"] == pais]
    if len(i):
        plt.annotate(pais, P[i[0]], fontsize=10, fontweight="bold")
v = pca.explained_variance_ratio_ * 100
plt.xlabel(f"Componente 1 ({v[0]:.0f} % de la varianza)"); plt.ylabel(f"Componente 2 ({v[1]:.0f} %)")
plt.legend(); plt.title("193 países agrupados por K-Means"); plt.show()

## Validar con lo que dejamos por fuera
Ahora sí miramos el IDH por grupo. Si los grupos tienen sentido, el IDH debería diferenciarse… o mostrarnos algo que no esperábamos.

In [ ]:
perfil = df.groupby("grupo").agg(paises=("country", "count"), idh_promedio=("hdi", "mean"),
                                  pib_pc=("gdp_per_capita_usd", "median"), esperanza_vida=("life_expectancy", "mean"),
                                  manufactura_pct=("manufacturing_pct_gdp", "mean")).round(2)
perfil

In [ ]:
# Ejemplos de países en cada grupo
for g, sub in df.groupby("grupo"):
    print(f"Grupo {g}:", ", ".join(sub["country"].head(8)))

> **Tip:** los grupos no traen nombre. Ponérselo es trabajo de investigación: mira el perfil de cada uno y nómbralo con teoría, no con el número del cluster.

## Para seguir con IA
> Te paso la tabla de perfiles de 4 grupos de países encontrados con K-Means. Propón un nombre para cada grupo, justifícalo con las variables y dime qué información adicional necesitaría para confirmar esa interpretación.

*Fuente de los datos: Banco Mundial (WDI y Worldwide Governance Indicators) y PNUD (IDH). Elaboración propia. Los resultados pueden diferir levemente de los de la presentación, que se hicieron en R.*